# 06 · The same numbers, recomputed in SQL

Part 6 of 6, a cross-check. The five earlier notebooks compute everything in Python. The raw material of this project is tabular, though: daily prices, a log of trades, a list of stress windows. Those are questions SQL answers directly, so I wrote the key figures a second time as queries and compared the two.

The point is not that SQL is better here. It is that two independent implementations agreeing to rounding error is a stronger check than a unit test written by the same person in the same language. The queries are plain text files in the `sql/` folder and run on DuckDB, a database that lives inside the Python process, so there is nothing to install or start.

Only the parts that are naturally tabular are redone. The HAR and GARCH fits, the option pricing and the backtest loop stay in Python. They are numerical problems, and forcing them into SQL would be a stunt, not an improvement.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from vrp.backtest import StraddleConfig, backtest_short_straddle
from vrp.data import load_market_data
from vrp.metrics import stress_table
from vrp.pipeline import volatility_premium
from vrp.sql import connect, read_query, run
from vrp.volatility import forward_realized_variance, log_returns

END = "2026-09-25"                   # last complete session used for the published results
EVAL_START = "2020-01-01"

data = load_market_data("yahoo", "1990-01-01", END)
spx, vix = data["spx"], data["vix"]
_, trades = backtest_short_straddle(spx.loc[EVAL_START:], vix.loc[EVAL_START:], config=StraddleConfig())

con = connect(spx, vix, trades)      # tables: market, trades, episodes; view: daily
print(f"{len(spx):,} trading days and {len(trades)} trades loaded")

9,251 trading days and 80 trades loaded


## Forward realized variance, as a window function

In [2]:
display(Markdown("```sql\n" + read_query("01_daily") + "```"))

```sql
-- Daily log returns and the forward 21-day realized variance, annualized.
--
--   RV(t, t+21) = 252 / 21 * sum_{i=1..21} r(t+i)^2
--
-- The frame looks at the 21 rows after the current one. The count guard leaves the last
-- 21 days empty, exactly as the Python version does, instead of averaging fewer returns.
CREATE OR REPLACE VIEW daily AS
WITH returns AS (
    SELECT day, spx, vix, LN(spx / LAG(spx) OVER (ORDER BY day)) AS r
    FROM market
),
squared AS (
    SELECT day, spx, vix, r, r * r AS r2
    FROM returns
    WHERE r IS NOT NULL
)
SELECT
    day,
    spx,
    vix,
    r,
    CASE WHEN COUNT(r2) OVER next_21 = 21 THEN 252 * AVG(r2) OVER next_21 END AS forward_variance
FROM squared
WINDOW next_21 AS (ORDER BY day ROWS BETWEEN 1 FOLLOWING AND 21 FOLLOWING);
```

This is the one place where SQL is neat. The window looks at the 21 rows after the current one, and the count guard leaves the last 21 days empty instead of averaging fewer returns, which is the definition of the Python function.

In [3]:
expected = forward_realized_variance(log_returns(spx))
sql_daily = con.execute("SELECT day, forward_variance FROM daily ORDER BY day").df().set_index("day")["forward_variance"]

days = pd.DataFrame({
    "Python": [len(expected), int(expected.isna().sum())],
    "SQL": [len(sql_daily), int(sql_daily.isna().sum())],
}, index=["days", "days without a value"])
largest = float(np.nanmax(np.abs(expected.to_numpy() - sql_daily.to_numpy())))
display(days)
display(Markdown(f"Largest gap between the two series: **{largest:.1e}**."))

,Python,SQL
days,9250,9250
days without a value,21,21


Largest gap between the two series: **4.6e-15**.

## The headline figures of the premium

In [4]:
sql_summary = run(con, "02_premium_summary", start=EVAL_START).iloc[0]
python_summary = volatility_premium(vix.loc[EVAL_START:], expected.loc[EVAL_START:])

comparison = pd.DataFrame({"SQL": sql_summary, "Python": python_summary})
comparison["difference"] = comparison["SQL"] - comparison["Python"]
comparison.round(4).assign(difference=comparison["difference"].map(lambda x: f"{x:.1e}"))

,SQL,Python,difference
mean_implied_vol,0.2078,0.2078,0.0e+00
mean_realized_vol,0.1695,0.1695,-1.4e-16
mean_spread_vol_pts,3.8284,3.8284,3.6e-15
share_positive,0.8486,0.8486,0.0e+00
worst_spread_vol_pts,-71.9626,-71.9626,4.3e-14


## Year by year

In [5]:
by_year_sql = run(con, "03_premium_by_year", start=EVAL_START).set_index("year")
by_year_sql.round(2)

,average_vix,average_realized_vol,gap_vol_points,share_of_days_vix_above,worst_gap_vol_points
year,,,,,
2020,29.25,27.09,2.16,0.77,-71.96
2021,19.66,12.79,6.87,0.95,-2.05
2022,25.62,23.74,1.89,0.65,-14.58
2023,16.87,12.55,4.32,1.00,0.11
2024,15.61,12.43,3.18,0.81,-8.46
2025,18.96,15.44,3.52,0.84,-31.10
2026,18.66,13.23,5.43,0.96,-1.64


In [6]:
gap = (vix / 100 - np.sqrt(expected)).dropna().loc[EVAL_START:]
year = gap.index.year
by_year_python = pd.DataFrame({
    "average_vix": vix.loc[gap.index].groupby(year).mean(),
    "average_realized_vol": (100 * np.sqrt(expected).loc[gap.index]).groupby(year).mean(),
    "gap_vol_points": (100 * gap).groupby(year).mean(),
    "share_of_days_vix_above": (gap > 0).groupby(year).mean(),
    "worst_gap_vol_points": (100 * gap).groupby(year).min(),
})
by_year_python.index.name = "year"
(by_year_sql - by_year_python).abs().max().map(lambda x: f"{x:.1e}").to_frame("largest gap, SQL vs Python")

,"largest gap, SQL vs Python"
average_vix,0.0e+00
average_realized_vol,2.1e-13
gap_vol_points,2.1e-13
share_of_days_vix_above,0.0e+00
worst_gap_vol_points,4.3e-13


## The trades

In [7]:
sql_trades = run(con, "04_trade_summary").iloc[0]
returns = trades["return_on_capital"]
python_trades = pd.Series({
    "trades": len(trades),
    "win_rate": (returns > 0).mean(),
    "average_win": returns[returns > 0].mean(),
    "average_loss": returns[returns <= 0].mean(),
    "best_trade": returns.max(),
    "worst_trade": returns.min(),
})
pd.DataFrame({"SQL": sql_trades, "Python": python_trades}).round(4)

,SQL,Python
trades,80.0000,80.0000
win_rate,0.7000,0.7000
average_win,0.0097,0.0097
average_loss,-0.0116,-0.0116
best_trade,0.0244,0.0244
worst_trade,-0.0822,-0.0822


In [8]:
run(con, "05_worst_trades").round({"implied_vol": 3, "realized_vol": 3, "pnl_pct": 2}).set_index("entry")

,expiry,implied_vol,realized_vol,pnl_pct
entry,,,,
2020-03-04,2020-04-02,0.300,0.934,-8.22
2020-02-03,2020-03-04,0.160,0.336,-3.43
2025-03-10,2025-04-08,0.259,0.318,-2.57
2024-07-08,2024-08-06,0.104,0.197,-1.95
2022-06-02,2022-07-05,0.227,0.290,-1.18


## The stress windows, as a join

In [9]:
sql_stress = run(con, "06_stress_episodes").set_index("episode")
python_stress = stress_table({"s": pd.Series(0.0, index=spx.index)}, spx, vix)[["spx_return", "vix_peak"]].astype(float)
display(sql_stress.assign(spx_return=(100 * sql_stress["spx_return"]).round(1)).rename(columns={"spx_return": "S&P 500 (%)", "vix_peak": "VIX peak"}).round(1))

,S&P 500 (%),VIX peak
episode,,
Covid crash 2020,-14.0,82.7
Tech selloff Sep 2020,-8.7,40.3
Retail squeeze Jan 2021,0.8,37.2
Omicron Nov-Dec 2021,-2.5,31.1
Rate shock 2022,-19.3,36.5
Regional banks Mar 2023,2.9,26.5
Yield spike Oct 2023,-5.8,21.7
Yen carry unwind Aug 2024,0.6,38.6
Tariff shock Apr 2025,-1.1,52.3


## All the checks together

In [10]:
checks = pd.Series({
    "forward realized variance": float(np.nanmax(np.abs(expected.to_numpy() - sql_daily.to_numpy()))),
    "premium headline figures": float(comparison["difference"].abs().max()),
    "premium by year": float((by_year_sql - by_year_python).abs().to_numpy().max()),
    "trade summary": float((sql_trades.astype(float) - python_trades.astype(float)).abs().max()),
    "stress windows": float((sql_stress - python_stress).abs().to_numpy().max()),
}, name="largest gap, SQL vs Python")
assert (checks < 1e-9).all(), checks
checks.map(lambda x: f"{x:.1e}").to_frame()

,"largest gap, SQL vs Python"
forward realized variance,4.6e-15
premium headline figures,4.3e-14
premium by year,4.3e-13
trade summary,3.5e-18
stress windows,0.0e+00


Every figure agrees to rounding error, the largest gap being about 4e-13. The SQL layer did not find a mistake in the Python results, and that is the useful outcome: the definitions were written twice, in two languages, and they say the same thing.

Where SQL felt natural: the window over the next 21 days, the average win and loss with `FILTER`, and the join between the stress windows and the daily table, which is a single query instead of a loop. Where it added nothing: the yearly table is about as short in pandas as in SQL. For this project I would keep Python as the working language and treat the queries as a validation layer, and as a readable statement of what each number means.